# Multimodal Moroccan Sign Language Translation (SLT)

**Pipeline:** Video → OpenPose → RGB features → Multimodal Fusion → LLM+LoRA → Arabic/French text + Annotated video

| Section | Content |
|---|---|
| 1 | Setup & imports |
| 2 | Dataset loading |
| 3 | OpenPose visualization |
| 4 | RGB visualization |
| 5 | Skeleton encoder (LSTM/Transformer) |
| 6 | RGB encoder (ResNet18 / ViT-tiny via timm) |
| 7 | Multimodal fusion + projection |
| 8 | LoRA integration |
| 9 | Qwen2.5-3B training |
| 10 | JAIS-13B training |
| 11 | Metrics (BLEU / ROUGE / METEOR / BERTScore / SacreBLEU / ChrF) |
| 12 | Inference pipeline |
| 13 | Translation generation |
| 14 | Educational template formatting |
| 15 | Checkpoint saving |
| 16 | Ablation study |
| 17 | Video rendering (skeleton overlay + text) |

## 1. Setup & Imports

In [8]:
# ── Dependencies ─────────────────────────────────────────────────────────
# timm 0.9.x works without torchvision. 1.x requires torchvision::nms which
# is absent in this environment — pin to 0.9.16.
import subprocess, sys
needed = [
    'timm==0.9.16',          # must be 0.9.x — 1.x requires torchvision
    'sacrebleu>=2.3.1',
    'rouge-score>=0.1.2',
    'nltk>=3.8.1',
    'bert-score>=0.3.13',
    'arabic-reshaper>=3.0.0',
    'python-bidi>=0.4.2',
    'peft>=0.10.0',
    'accelerate>=0.29.0',
    'bitsandbytes>=0.43.0',
    'einops>=0.7.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + needed)
print('Dependencies ready.')

# ── Standard imports ──────────────────────────────────────────────────────
from __future__ import annotations
import json, math, os, sys, unicodedata, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import timm

import arabic_reshaper
from bidi.algorithm import get_display

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

warnings.filterwarnings('ignore')

# ── Repo root ─────────────────────────────────────────────────────────────
REPO_ROOT = Path('.')
for _p in [Path('.'), Path('..'), Path('/workspaces/Master_Multimodal-Moroccan-SLG-main')]:
    if (_p / 'mosl').is_dir():
        REPO_ROOT = _p.resolve(); break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mosl.data.dataset import MoSLSkelsDataset, mosl_collate, COORDS_PER_FRAME
from mosl.text.tokenizer import WordTokenizer
from mosl.render.pose_bridge import (
    draw_pose_frame, COCO18_LIMBS, COCO18_COLORS, HAND_EDGES, CONF_THR,
)

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR       = REPO_ROOT / 'data'
PROCESSED_DIR  = DATA_DIR / 'processed'
VOCAB_PATH     = PROCESSED_DIR / 'vocab.json'
LABELS_CSV     = DATA_DIR / 'labels.csv'
FINAL_DATA_DIR = REPO_ROOT / 'third_party' / 'Prompt2Sign' / 'tools' / '2D_to_3D' / 'final_data'
OUTPUTS_DIR    = REPO_ROOT / 'outputs' / 'slt'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch  : {torch.__version__}')
print(f'timm   : {timm.__version__}')
print(f'device : {DEVICE}')
print(f'repo   : {REPO_ROOT}')



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Dependencies ready.


RuntimeError: operator torchvision::nms does not exist

## 2. Dataset Loading

In [ ]:
# ── SLT Dataset ──────────────────────────────────────────────────────────
# Wraps MoSLSkelsDataset (pose) and adds RGB frame loading from raw videos.
# Format: { pose_sequence, rgb_frames, target_text }
# All target_text values come exclusively from real dataset annotations.

class SLTDataset(Dataset):
    def __init__(self, mode, tokenizer, final_data_dir=None,
                 video_root=None, max_rgb_frames=16, rgb_size=224,
                 repo_root=None):
        self.mode = mode
        self.tokenizer = tokenizer
        self.max_rgb_frames = max_rgb_frames
        self.rgb_size = rgb_size
        repo_root = repo_root or REPO_ROOT
        self.video_root = video_root or (DATA_DIR / 'raw' / 'vedios-dataset')
        self._ds = MoSLSkelsDataset(
            mode, tokenizer=tokenizer,
            final_data_dir=final_data_dir or FINAL_DATA_DIR,
            repo_root=repo_root,
        )

    def __len__(self): return len(self._ds)

    def _load_rgb(self, clip_id):
        T, H, W = self.max_rgb_frames, self.rgb_size, self.rgb_size
        blank = np.zeros((T, H, W, 3), dtype=np.uint8)
        hits = list(self.video_root.rglob(f'{clip_id}.mp4'))
        if not hits:
            return blank
        cap = cv2.VideoCapture(str(hits[0]))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release(); return blank
        idxs = np.linspace(0, total - 1, T, dtype=int)
        frames = []
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, f = cap.read()
            if not ok:
                f = np.zeros((H, W, 3), dtype=np.uint8)
            else:
                f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                f = cv2.resize(f, (W, H))
            frames.append(f)
        cap.release()
        return np.stack(frames)

    def __getitem__(self, idx):
        s = self._ds[idx]
        text = self.tokenizer.decode(s['text_ids'].tolist(), strip_specials=True)
        rgb  = self._load_rgb(s['clip_id'])
        return {
            'pose_sequence': s['pose'],
            'rgb_frames':    torch.from_numpy(rgb),
            'target_text':   text,
            'text_ids':      s['text_ids'],
            'time':          s['time'],
            'n_frames':      s['n_frames'],
            'clip_id':       s['clip_id'],
        }


def slt_collate(batch):
    B = len(batch)
    T_max = max(s['n_frames'] for s in batch)
    T_rgb = batch[0]['rgb_frames'].shape[0]
    H = batch[0]['rgb_frames'].shape[1]
    W = batch[0]['rgb_frames'].shape[2]
    pose      = torch.zeros(B, T_max, COORDS_PER_FRAME)
    time_t    = torch.zeros(B, T_max)
    pose_mask = torch.zeros(B, T_max, dtype=torch.bool)
    rgb       = torch.zeros(B, T_rgb, H, W, 3, dtype=torch.uint8)
    n_frames  = torch.empty(B, dtype=torch.long)
    texts, clip_ids = [], []
    for b, s in enumerate(batch):
        T = s['n_frames']
        pose[b, :T]      = s['pose_sequence']
        time_t[b, :T]    = s['time']
        pose_mask[b, :T] = True
        rgb[b]           = s['rgb_frames']
        n_frames[b]      = T
        texts.append(s['target_text'])
        clip_ids.append(s['clip_id'])
    return dict(pose=pose, pose_mask=pose_mask, time=time_t,
                rgb_frames=rgb, n_frames=n_frames,
                target_text=texts, clip_ids=clip_ids)


# ── Load ──────────────────────────────────────────────────────────────────
if VOCAB_PATH.exists():
    word_tok = WordTokenizer.load(VOCAB_PATH)
    print(f'Vocab: {word_tok.vocab_size} tokens ({word_tok.n_signs} signs)')
else:
    print(f'[WARN] vocab.json not found at {VOCAB_PATH}')
    word_tok = None

if word_tok and FINAL_DATA_DIR.exists():
    train_ds = SLTDataset('train', word_tok)
    dev_ds   = SLTDataset('dev',   word_tok)
    test_ds  = SLTDataset('test',  word_tok)
    print(f'Train {len(train_ds)} | Dev {len(dev_ds)} | Test {len(test_ds)}')
    SAMPLE = train_ds[0]
else:
    print('[INFO] Dataset not found — demo mode.')
    train_ds = dev_ds = test_ds = None
    SAMPLE = {
        'pose_sequence': torch.zeros(60, 150),
        'rgb_frames':    torch.zeros(16, 224, 224, 3, dtype=torch.uint8),
        'target_text':   'أَنَا',
        'clip_id':       'demo',
        'n_frames':      60,
        'time':          torch.linspace(0.01, 1.0, 60),
        'text_ids':      torch.tensor([1, 4, 2]),
    }
print('Sample clip_id:', SAMPLE['clip_id'])
print('  pose_sequence:', tuple(SAMPLE['pose_sequence'].shape))
print('  rgb_frames   :', tuple(SAMPLE['rgb_frames'].shape))
print('  target_text  :', repr(SAMPLE['target_text']))


## 3. OpenPose Visualization

In [ ]:
def _ar(t): return get_display(arabic_reshaper.reshape(t))

def viz_openpose(pose_seq, title='', n=6, canvas=256):
    T = pose_seq.shape[0]
    idxs = np.linspace(0, T-1, min(n, T), dtype=int)
    fig, axes = plt.subplots(1, len(idxs), figsize=(3*len(idxs), 3.5))
    if len(idxs) == 1: axes = [axes]
    for ax, t in zip(axes, idxs):
        joints = pose_seq[t].numpy().reshape(50, 3)
        xy = joints[:18, :2].copy()
        mn, mx = xy.min(), xy.max()
        span = max(mx - mn, 1e-6)
        xy = (xy - mn) / span * (canvas * 0.8) + canvas * 0.1
        conf = (joints[:18, :2].sum(1) != 0).astype(np.float32)
        body = np.concatenate([xy, conf[:, None]], 1)
        def _hand(s):
            h = joints[s:s+21, :2].copy()
            h = (h - mn) / span * (canvas * 0.8) + canvas * 0.1
            c = (joints[s:s+21, :2].sum(1) != 0).astype(np.float32)
            return np.concatenate([h, c[:, None]], 1)
        hl = _hand(18) if joints.shape[0] >= 39 else np.zeros((21,3))
        hr = _hand(39) if joints.shape[0] >= 60 else np.zeros((21,3))
        img = draw_pose_frame(body, hl, hr, np.zeros((0,3)), canvas, False)
        ax.imshow(np.array(img)); ax.set_title(f't={t}', fontsize=9); ax.axis('off')
    if title: fig.suptitle(_ar(title), fontsize=12)
    plt.tight_layout()
    out = OUTPUTS_DIR / 'openpose_viz.png'
    plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
    print('Saved ->', out)

viz_openpose(SAMPLE['pose_sequence'], title=SAMPLE['target_text'])


## 4. RGB Visualization

In [ ]:
def viz_rgb(rgb_frames, title='', n=8):
    T = rgb_frames.shape[0]
    idxs = np.linspace(0, T-1, min(n, T), dtype=int)
    arr = rgb_frames.numpy()
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.5*len(idxs), 3))
    if len(idxs) == 1: axes = [axes]
    for ax, t in zip(axes, idxs):
        ax.imshow(arr[t]); ax.set_title(f't={t}', fontsize=9); ax.axis('off')
    if title: fig.suptitle(_ar(title), fontsize=12)
    plt.tight_layout()
    out = OUTPUTS_DIR / 'rgb_viz.png'
    plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
    print('Saved ->', out)

viz_rgb(SAMPLE['rgb_frames'], title=SAMPLE['target_text'])


## 5. Skeleton Encoder

In [ ]:
# Temporal Transformer encoder for OpenPose keypoint sequences.
# Input : (B, T, 150)  Output: (B, d_model)

class SkeletonEncoder(nn.Module):
    def __init__(self, pose_dim=150, d_model=512, nhead=8,
                 n_layers=4, d_ff=2048, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.proj = nn.Linear(pose_dim, d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        enc = nn.TransformerEncoderLayer(d_model, nhead, d_ff, dropout,
                                         activation='relu', batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, pose, mask):
        x = self.proj(pose) * math.sqrt(self.d_model)
        x = x + self.pe[:, :x.size(1)]
        x = self.encoder(x, src_key_padding_mask=~mask)
        mf = mask.unsqueeze(-1).float()
        return self.norm((x * mf).sum(1) / mf.sum(1).clamp(min=1))


skel_enc = SkeletonEncoder().to(DEVICE)
print(f'SkeletonEncoder: {sum(p.numel() for p in skel_enc.parameters()):,} params')
_p = SAMPLE['pose_sequence'].unsqueeze(0).to(DEVICE)
_m = torch.ones(1, _p.shape[1], dtype=torch.bool).to(DEVICE)
_se = skel_enc(_p, _m)
print('Output:', tuple(_se.shape))


## 6. RGB Encoder (ResNet18 / ViT-tiny)

In [ ]:
# RGB encoder using timm (no torchvision dependency).
# Supports 'resnet18' (512-d) and 'vit_tiny_patch16_224' (192-d).
# Input : (B, T, H, W, 3) uint8   Output: (B, d_model)

class RGBEncoder(nn.Module):
    def __init__(self, backbone='resnet18', d_model=512,
                 freeze_backbone=True, pretrained=False):
        super().__init__()
        self.d_model = d_model
        # timm: num_classes=0 removes the classifier head
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained,
            num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)
        self.proj = nn.Linear(feat_dim, d_model)
        self.norm = nn.LayerNorm(d_model)
        # ImageNet normalisation
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def forward(self, rgb):  # (B, T, H, W, 3) uint8
        B, T, H, W, C = rgb.shape
        x = rgb.view(B*T, H, W, C).float() / 255.0
        x = x.permute(0, 3, 1, 2)           # (B*T, 3, H, W)
        x = (x - self.mean) / self.std
        x = self.backbone(x)                 # (B*T, feat_dim)
        x = self.proj(x).view(B, T, -1).mean(1)  # (B, d_model)
        return self.norm(x)


# Default: ResNet18 (pretrained=False avoids download; set True when online)
RGB_BACKBONE = 'resnet18'   # or 'vit_tiny_patch16_224'
rgb_enc = RGBEncoder(backbone=RGB_BACKBONE, pretrained=False).to(DEVICE)
trainable = sum(p.numel() for p in rgb_enc.parameters() if p.requires_grad)
print(f'RGBEncoder ({RGB_BACKBONE}): {trainable:,} trainable params')
_r = SAMPLE['rgb_frames'].unsqueeze(0).to(DEVICE)
_re = rgb_enc(_r)
print('Output:', tuple(_re.shape))


## 7. Multimodal Fusion + Projection

In [ ]:
# Cross-attention fusion: skeleton queries attend over RGB keys/values.
# Output: (B, n_prefix, llm_hidden) — soft prefix tokens for the LLM.

@dataclass
class FusionConfig:
    skel_dim: int = 512
    rgb_dim:  int = 512
    d_model:  int = 512
    llm_hidden: int = 3072   # Qwen2.5-3B; override to 5120 for JAIS-13B
    n_prefix:   int = 8
    nhead:      int = 8
    dropout:    float = 0.1


class MultimodalFusion(nn.Module):
    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.skel_proj = nn.Linear(cfg.skel_dim, d)
        self.rgb_proj  = nn.Linear(cfg.rgb_dim,  d)
        self.cross = nn.MultiheadAttention(d, cfg.nhead,
                                           dropout=cfg.dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d*2, d*4), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(d*4, d),
        )
        self.to_prefix = nn.Linear(d, cfg.n_prefix * cfg.llm_hidden)

    def forward(self, skel_emb, rgb_emb):
        B = skel_emb.size(0)
        s = self.skel_proj(skel_emb).unsqueeze(1)   # (B,1,d)
        r = self.rgb_proj(rgb_emb).unsqueeze(1)     # (B,1,d)
        attn, _ = self.cross(s, r, r)
        s = self.norm1(s + attn)
        r = self.norm2(r)
        fused = self.ff(torch.cat([s, r], -1))      # (B,1,d)
        return self.to_prefix(fused.squeeze(1)).view(B, self.cfg.n_prefix, self.cfg.llm_hidden)


fusion_cfg = FusionConfig()
fusion = MultimodalFusion(fusion_cfg).to(DEVICE)
print(f'Fusion: {sum(p.numel() for p in fusion.parameters()):,} params')
_pf = fusion(_se, _re)
print('Prefix shape:', tuple(_pf.shape))  # (1, 8, 3072)


## 8. LoRA Integration & SLTModel

In [ ]:
@dataclass
class LoRACfg:
    r: int = 16
    alpha: int = 32
    dropout: float = 0.05
    bias: str = 'none'
    targets_qwen: list = field(default_factory=lambda: [
        'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    targets_jais: list = field(default_factory=lambda: ['c_attn','c_proj','c_fc'])


class SLTModel(nn.Module):
    """Full SLT model: encoders + fusion + LLM+LoRA.
    Only LoRA weights, encoders, and fusion are trained.
    All labels come from real dataset annotations.
    """
    def __init__(self, llm_name, lora_cfg: LoRACfg, fusion_cfg: FusionConfig,
                 load_in_4bit=True):
        super().__init__()
        self.llm_name = llm_name
        self.skel_enc = SkeletonEncoder(d_model=fusion_cfg.skel_dim)
        self.rgb_enc  = RGBEncoder(backbone=RGB_BACKBONE, d_model=fusion_cfg.rgb_dim)
        self.fusion   = MultimodalFusion(fusion_cfg)

        bnb = None
        if load_in_4bit and torch.cuda.is_available():
            bnb = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
        self.tok = AutoTokenizer.from_pretrained(llm_name, trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_name, quantization_config=bnb, trust_remote_code=True,
            device_map='auto' if torch.cuda.is_available() else None,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        is_jais = 'jais' in llm_name.lower()
        peft_cfg = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=lora_cfg.r, lora_alpha=lora_cfg.alpha,
            lora_dropout=lora_cfg.dropout, bias=lora_cfg.bias,
            target_modules=lora_cfg.targets_jais if is_jais else lora_cfg.targets_qwen,
        )
        self.llm = get_peft_model(self.llm, peft_cfg)
        self.llm.print_trainable_parameters()

    def _embed_fn(self):
        return self.llm.base_model.model.model.embed_tokens

    def _encode_visual(self, pose, pose_mask, rgb):
        se = self.skel_enc(pose, pose_mask)
        re = self.rgb_enc(rgb.float().to(pose.device))
        return self.fusion(se, re)   # (B, n_prefix, llm_hidden)

    def forward(self, pose, pose_mask, rgb, target_text):
        B = pose.size(0)
        prefix = self._encode_visual(pose, pose_mask, rgb)
        prompt = 'ترجم تسلسل لغة الإشارة المغربية إلى نص عربي.\nالترجمة: '
        full   = [prompt + t for t in target_text]
        enc = self.tok(full, return_tensors='pt', padding=True,
                       truncation=True, max_length=128).to(pose.device)
        te = self._embed_fn()(enc['input_ids'])          # (B, L, h)
        ie = torch.cat([prefix, te], 1)                  # (B, n+L, h)
        pm = torch.ones(B, prefix.size(1), dtype=torch.long, device=pose.device)
        am = torch.cat([pm, enc['attention_mask']], 1)
        pl = torch.full((B, prefix.size(1)), -100, dtype=torch.long, device=pose.device)
        lb = torch.cat([pl, enc['input_ids']], 1)
        lb[lb == self.tok.pad_token_id] = -100
        return self.llm(inputs_embeds=ie, attention_mask=am, labels=lb).loss

    @torch.no_grad()
    def translate(self, pose, pose_mask, rgb, max_new=64):
        B = pose.size(0)
        prefix = self._encode_visual(pose, pose_mask, rgb)
        prompt = 'ترجم تسلسل لغة الإشارة المغربية إلى نص عربي.\nالترجمة:'
        enc = self.tok(prompt, return_tensors='pt').to(pose.device)
        te  = self._embed_fn()(enc['input_ids'])
        ie  = torch.cat([prefix, te], 1)
        pm  = torch.ones(B, prefix.size(1), dtype=torch.long, device=pose.device)
        am  = torch.cat([pm, enc['attention_mask']], 1)
        ids = self.llm.generate(
            inputs_embeds=ie, attention_mask=am,
            max_new_tokens=max_new, do_sample=False,
            pad_token_id=self.tok.eos_token_id,
        )
        return self.tok.decode(ids[0, ie.size(1):], skip_special_tokens=True).strip()


print('SLTModel defined.')
print('Instantiate: model = SLTModel("Qwen/Qwen2.5-3B-Instruct", LoRACfg(), FusionConfig())')


## 9. Qwen2.5-3B-Instruct Training

In [ ]:
QWEN_ID  = 'Qwen/Qwen2.5-3B-Instruct'
QWEN_DIR = OUTPUTS_DIR / 'qwen_lora'
QWEN_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class TrainCfg:
    model_name:  str  = QWEN_ID
    output_dir:  Path = QWEN_DIR
    batch_size:  int  = 4
    grad_accum:  int  = 4
    lr:          float = 2e-4
    weight_decay:float = 0.01
    max_epochs:  int  = 10
    warmup_ratio:float = 0.05
    max_grad_norm:float = 1.0
    eval_every:  int  = 100
    save_every:  int  = 200
    seed:        int  = 42


def _save_ckpt(model, out_dir, metrics=None, history=None):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    model.llm.save_pretrained(str(out_dir / 'lora_adapter'))
    model.tok.save_pretrained(str(out_dir / 'tokenizer'))
    torch.save({
        'skel_enc': model.skel_enc.state_dict(),
        'rgb_enc':  model.rgb_enc.state_dict(),
        'fusion':   model.fusion.state_dict(),
        'llm_name': model.llm_name,
    }, out_dir / 'encoders.pt')
    if metrics:
        json.dump(metrics, open(out_dir/'metrics.json','w',encoding='utf-8'),
                  ensure_ascii=False, indent=2)
    if history:
        json.dump(history, open(out_dir/'history.json','w',encoding='utf-8'),
                  ensure_ascii=False, indent=2)
    print('Checkpoint ->', out_dir)


@torch.no_grad()
def _eval_loss(model, loader):
    model.eval()
    tot, n = 0.0, 0
    for b in loader:
        loss = model(b['pose'].to(DEVICE), b['pose_mask'].to(DEVICE),
                     b['rgb_frames'].to(DEVICE), b['target_text'])
        tot += loss.item(); n += 1
    return tot / max(n, 1)


def train_slt(model, train_ds, dev_ds, cfg: TrainCfg):
    torch.manual_seed(cfg.seed)
    tr_dl = DataLoader(train_ds, cfg.batch_size, shuffle=True,
                       collate_fn=slt_collate, num_workers=2, pin_memory=True)
    dv_dl = DataLoader(dev_ds,   cfg.batch_size, shuffle=False,
                       collate_fn=slt_collate, num_workers=2, pin_memory=True)
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=cfg.lr, weight_decay=cfg.weight_decay)
    total = len(tr_dl) * cfg.max_epochs // cfg.grad_accum
    sched = get_cosine_schedule_with_warmup(
        opt, int(total * cfg.warmup_ratio), total)
    hist = {'step': [], 'train_loss': [], 'dev_loss': []}
    best, step = float('inf'), 0
    model.train(); opt.zero_grad()
    for epoch in range(cfg.max_epochs):
        for i, batch in enumerate(tqdm(tr_dl, desc=f'Epoch {epoch+1}')):
            loss = model(batch['pose'].to(DEVICE), batch['pose_mask'].to(DEVICE),
                         batch['rgb_frames'].to(DEVICE), batch['target_text'])
            (loss / cfg.grad_accum).backward()
            if (i+1) % cfg.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                opt.step(); sched.step(); opt.zero_grad(); step += 1
                if step % cfg.eval_every == 0:
                    dl = _eval_loss(model, dv_dl)
                    hist['step'].append(step)
                    hist['train_loss'].append(loss.item())
                    hist['dev_loss'].append(dl)
                    print(f'  step {step}: train={loss.item():.4f} dev={dl:.4f}')
                    model.train()
                    if dl < best:
                        best = dl
                        _save_ckpt(model, cfg.output_dir / 'best')
                if step % cfg.save_every == 0:
                    _save_ckpt(model, cfg.output_dir / f'step_{step}')
    _save_ckpt(model, cfg.output_dir / 'final', history=hist)
    return hist


print('Training utilities ready.')
print('To train Qwen2.5-3B:')
print('  model = SLTModel(QWEN_ID, LoRACfg(), FusionConfig(llm_hidden=3072))')
print('  hist  = train_slt(model, train_ds, dev_ds, TrainCfg())')
# Uncomment when dataset + GPU available:
# model_qwen = SLTModel(QWEN_ID, LoRACfg(), FusionConfig(llm_hidden=3072))
# hist_qwen  = train_slt(model_qwen, train_ds, dev_ds, TrainCfg())


## 10. JAIS-13B Training

In [ ]:
JAIS_ID  = 'inceptionai/jais-13b'
JAIS_DIR = OUTPUTS_DIR / 'jais_lora'
JAIS_DIR.mkdir(parents=True, exist_ok=True)
JAIS_HIDDEN = 5120  # JAIS-13B hidden size

print('To train JAIS-13B (requires ~26 GB VRAM in 4-bit):')
print('  jais_fusion = FusionConfig(llm_hidden=JAIS_HIDDEN)')
print('  model_jais  = SLTModel(JAIS_ID, LoRACfg(), jais_fusion)')
print('  hist_jais   = train_slt(model_jais, train_ds, dev_ds,')
print('                  TrainCfg(model_name=JAIS_ID, output_dir=JAIS_DIR,')
print('                           batch_size=2, grad_accum=8))')

# Uncomment when dataset + GPU available:
# jais_fusion = FusionConfig(llm_hidden=JAIS_HIDDEN)
# model_jais  = SLTModel(JAIS_ID, LoRACfg(), jais_fusion)
# hist_jais   = train_slt(
#     model_jais, train_ds, dev_ds,
#     TrainCfg(model_name=JAIS_ID, output_dir=JAIS_DIR, batch_size=2, grad_accum=8)
# )


## 11. Evaluation Metrics

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import sacrebleu
from rouge_score import rouge_scorer as _rouge
from nltk.translate.meteor_score import meteor_score
from bert_score import score as _bert_score


def compute_metrics(hypotheses, references,
                    bert_model='aubmindlab/bert-base-arabertv2',
                    lang='ar'):
    """Compute BLEU-4, ROUGE-L, METEOR, BERTScore-F1, SacreBLEU, ChrF.
    All scores are on real dataset annotations only.
    """
    assert len(hypotheses) == len(references)

    # BLEU-4 (char tokenisation for Arabic)
    bleu = sacrebleu.corpus_bleu(
        hypotheses, [references], tokenize='char').score

    # SacreBLEU (13a tokenisation)
    sbleu = sacrebleu.corpus_bleu(
        hypotheses, [references], tokenize='13a').score

    # ChrF (character n-gram F-score)
    chrf = sacrebleu.corpus_chrf(hypotheses, [references]).score

    # ROUGE-L
    scorer = _rouge.RougeScorer(['rougeL'], use_stemmer=False)
    rouge_l = float(np.mean([
        scorer.score(r, h)['rougeL'].fmeasure
        for h, r in zip(hypotheses, references)
    ])) * 100

    # METEOR
    meteor = float(np.mean([
        meteor_score([r.split()], h.split())
        for h, r in zip(hypotheses, references)
    ])) * 100

    # BERTScore
    _, _, F1 = _bert_score(hypotheses, references,
                            model_type=bert_model, lang=lang, verbose=False)
    bscore = float(F1.mean()) * 100

    return {
        'bleu':         round(bleu,   2),
        'sacrebleu':    round(sbleu,  2),
        'chrf':         round(chrf,   2),
        'rouge_l':      round(rouge_l,2),
        'meteor':       round(meteor, 2),
        'bertscore_f1': round(bscore, 2),
    }


def print_metrics(m, label=''):
    if label: print(f'── {label} ──')
    for k, v in m.items():
        print(f'  {k:<18}: {v:.2f}')


# Quick sanity check on identical strings
demo = compute_metrics(['أَنَا', 'مَرْحَبًا'], ['أَنَا', 'مَرْحَبًا'])
print_metrics(demo, 'Demo (perfect match)')


## 12. Inference Pipeline

In [ ]:
def load_slt_model(ckpt_dir, llm_name, lora_cfg, fusion_cfg):
    """Reload a trained SLTModel from a checkpoint directory."""
    model = SLTModel(llm_name, lora_cfg, fusion_cfg, load_in_4bit=True)
    enc_path = Path(ckpt_dir) / 'encoders.pt'
    if enc_path.exists():
        st = torch.load(enc_path, map_location='cpu')
        model.skel_enc.load_state_dict(st['skel_enc'])
        model.rgb_enc.load_state_dict(st['rgb_enc'])
        model.fusion.load_state_dict(st['fusion'])
        print('Encoders loaded from', enc_path)
    adp = Path(ckpt_dir) / 'lora_adapter'
    if adp.exists():
        model.llm = PeftModel.from_pretrained(
            model.llm.base_model.model, str(adp))
        print('LoRA adapter loaded from', adp)
    model.to(DEVICE).eval()
    return model


def run_inference(model, dataset, indices, max_new=64):
    """Translate selected samples. Returns list of {clip_id, reference, hypothesis}."""
    results = []
    model.eval()
    for idx in tqdm(indices, desc='Inference'):
        s = dataset[idx]
        pose = s['pose_sequence'].unsqueeze(0).to(DEVICE)
        mask = torch.ones(1, pose.shape[1], dtype=torch.bool).to(DEVICE)
        rgb  = s['rgb_frames'].unsqueeze(0).to(DEVICE)
        hyp  = model.translate(pose, mask, rgb, max_new)
        results.append({
            'clip_id':    s['clip_id'],
            'reference':  s['target_text'],
            'hypothesis': hyp,
        })
    return results


print('Inference pipeline ready.')
print('Usage:')
print('  model = load_slt_model(QWEN_DIR/"best", QWEN_ID, LoRACfg(), FusionConfig())')
print('  results = run_inference(model, test_ds, list(range(20)))')


## 13. Translation Generation

In [ ]:
def generate_translations(model, test_ds, out_path, n=None):
    """Translate test set, compute metrics, save JSON."""
    n = len(test_ds) if n is None else min(n, len(test_ds))
    results = run_inference(model, test_ds, list(range(n)))
    hyps = [r['hypothesis'] for r in results]
    refs = [r['reference']  for r in results]
    metrics = compute_metrics(hyps, refs)
    print_metrics(metrics, f'{model.llm_name} — {n} test samples')
    out = {'model': model.llm_name, 'metrics': metrics, 'results': results}
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(out, open(out_path, 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)
    print('Results ->', out_path)
    return out


print('generate_translations() ready.')
print('Usage: output = generate_translations(model, test_ds,')
print('           OUTPUTS_DIR / "qwen_results.json")')


## 14. Educational Template Formatting

In [ ]:
TEMPLATES = [
    'هكذا نعبر عن {text} بلغة الإشارة المغربية',
    'الترجمة بلغة الإشارة المغربية: {text}',
    'هذا الفيديو يمثل الإشارة الخاصة بـ: {text}',
    'النص المترجم من لغة الإشارة المغربية هو: {text}',
]


def apply_template(text, idx=0):
    return TEMPLATES[idx % len(TEMPLATES)].format(text=text)


def format_results(results, out_path=None):
    """Add all template variants to each result entry."""
    out = []
    for r in results:
        e = dict(r)
        e['formatted'] = [apply_template(r['hypothesis'], i)
                          for i in range(len(TEMPLATES))]
        out.append(e)
    if out_path:
        json.dump(out, open(out_path, 'w', encoding='utf-8'),
                  ensure_ascii=False, indent=2)
        print('Formatted ->', out_path)
    return out


# Demo
for i, t in enumerate(TEMPLATES):
    print(f'[{i}]', apply_template('أَنَا', i))


## 15. Checkpoint Saving

In [ ]:
def save_checkpoint(model, out_dir, metrics=None, history=None, tag='final'):
    """Save LoRA adapter, tokenizer, encoders, metrics, history."""
    d = Path(out_dir) / tag
    d.mkdir(parents=True, exist_ok=True)
    model.llm.save_pretrained(str(d / 'lora_adapter'))
    model.tok.save_pretrained(str(d / 'tokenizer'))
    torch.save({
        'skel_enc': model.skel_enc.state_dict(),
        'rgb_enc':  model.rgb_enc.state_dict(),
        'fusion':   model.fusion.state_dict(),
        'llm_name': model.llm_name,
    }, d / 'encoders.pt')
    if metrics:
        json.dump(metrics, open(d/'metrics.json','w',encoding='utf-8'),
                  ensure_ascii=False, indent=2)
    if history:
        json.dump(history, open(d/'history.json','w',encoding='utf-8'),
                  ensure_ascii=False, indent=2)
    print('Saved ->', d)
    return d


def list_checkpoints(root):
    root = Path(root)
    ckpts = sorted(p for p in root.rglob('encoders.pt'))
    for c in ckpts:
        mp = c.parent / 'metrics.json'
        tag = f'  BLEU={json.load(open(mp))["bleu"]}' if mp.exists() else ''
        print(f'  {c.parent.relative_to(root)}{tag}')
    return ckpts


print('Checkpoint utilities ready.')
print('Existing checkpoints:')
list_checkpoints(OUTPUTS_DIR)


## 16. Ablation Study

In [ ]:
# Four ablation variants:
#   pose_only   — skeleton encoder only, no RGB
#   rgb_only    — RGB encoder only, no pose
#   fusion_qwen — both modalities, Qwen2.5-3B
#   fusion_jais — both modalities, JAIS-13B

@dataclass
class AblationVariant:
    name:      str
    use_pose:  bool = True
    use_rgb:   bool = True
    llm_name:  str  = QWEN_ID
    llm_hidden:int  = 3072


ABLATION_VARIANTS = [
    AblationVariant('pose_only',   use_pose=True,  use_rgb=False, llm_name=QWEN_ID),
    AblationVariant('rgb_only',    use_pose=False, use_rgb=True,  llm_name=QWEN_ID),
    AblationVariant('fusion_qwen', use_pose=True,  use_rgb=True,  llm_name=QWEN_ID),
    AblationVariant('fusion_jais', use_pose=True,  use_rgb=True,
                    llm_name=JAIS_ID, llm_hidden=JAIS_HIDDEN),
]


class AblationModel(SLTModel):
    """SLTModel that can zero-out one modality for ablation."""
    def __init__(self, v: AblationVariant, lora_cfg, fusion_cfg):
        super().__init__(v.llm_name, lora_cfg, fusion_cfg)
        self.use_pose = v.use_pose
        self.use_rgb  = v.use_rgb

    def _encode_visual(self, pose, mask, rgb):
        d = self.fusion.cfg.d_model
        se = self.skel_enc(pose, mask) if self.use_pose \
             else torch.zeros(pose.size(0), d, device=pose.device)
        re = self.rgb_enc(rgb.float().to(pose.device)) if self.use_rgb \
             else torch.zeros(pose.size(0), d, device=pose.device)
        return self.fusion(se, re)


def run_ablation(variants, train_ds, dev_ds, test_ds, base_cfg, lora_cfg):
    all_res = {}
    for v in variants:
        print(f'\n=== {v.name} ===')
        fc = FusionConfig(llm_hidden=v.llm_hidden)
        model = AblationModel(v, lora_cfg, fc).to(DEVICE)
        cfg = TrainCfg(model_name=v.llm_name,
                       output_dir=OUTPUTS_DIR/'ablation'/v.name,
                       batch_size=base_cfg.batch_size,
                       max_epochs=base_cfg.max_epochs)
        hist = train_slt(model, train_ds, dev_ds, cfg)
        n = min(50, len(test_ds))
        res = run_inference(model, test_ds, list(range(n)))
        m = compute_metrics([r['hypothesis'] for r in res],
                            [r['reference']  for r in res])
        print_metrics(m, v.name)
        all_res[v.name] = {'metrics': m, 'history': hist}
    sp = OUTPUTS_DIR / 'ablation_summary.json'
    json.dump(all_res, open(sp,'w',encoding='utf-8'), ensure_ascii=False, indent=2)
    print('Ablation summary ->', sp)
    return all_res


print('Ablation variants:', [v.name for v in ABLATION_VARIANTS])
print('To run: run_ablation(ABLATION_VARIANTS, train_ds, dev_ds, test_ds,')
print('            TrainCfg(max_epochs=5), LoRACfg())')
# Uncomment when ready:
# ablation_results = run_ablation(ABLATION_VARIANTS, train_ds, dev_ds, test_ds,
#                                 TrainCfg(max_epochs=5), LoRACfg())


## 17. Video Rendering (Skeleton Overlay + Translation Text)

In [ ]:
# ── Video rendering module ───────────────────────────────────────────────
# For each input video, produces an output video with:
#   1. Original RGB frame
#   2. OpenPose skeleton overlay (COCO-18 body + 21-pt hands)
#   3. Predicted Arabic translation text burned onto the frame
#
# Example overlay text:
#   'الترجمة بلغة الإشارة المغربية: أنا أحب المغرب'

from PIL import ImageFont

# COCO-18 limb pairs and BGR colours for OpenCV drawing
_LIMBS = [
    (1,2),(1,5),(2,3),(3,4),(5,6),(6,7),(1,8),(8,9),(9,10),
    (1,11),(11,12),(12,13),(1,0),(0,14),(14,16),(0,15),(15,17),
]
_COLORS_BGR = [
    (0,0,255),(0,85,255),(0,170,255),(0,255,255),(0,255,170),
    (0,255,85),(0,255,0),(85,255,0),(170,255,0),(255,255,0),
    (255,170,0),(255,85,0),(255,0,0),(85,0,255),(170,0,255),
    (255,0,255),(255,0,170),(255,0,85),
]
_HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
]


def _draw_skeleton_cv2(frame, pose_flat, conf_thr=0.1):
    """Draw COCO-18 body + hands on a BGR frame in-place. Returns frame."""
    H, W = frame.shape[:2]
    joints = np.array(pose_flat, dtype=np.float32).reshape(-1, 3)  # (K, x/y/c)
    body = joints[:18] if len(joints) >= 18 else np.zeros((18, 3))
    hl   = joints[18:39] if len(joints) >= 39 else np.zeros((21, 3))
    hr   = joints[39:60] if len(joints) >= 60 else np.zeros((21, 3))

    def pt(j):
        return (int(j[0]), int(j[1]))

    # Body limbs
    for li, (a, b) in enumerate(_LIMBS):
        if body[a,2] > conf_thr and body[b,2] > conf_thr:
            cv2.line(frame, pt(body[a]), pt(body[b]), _COLORS_BGR[li], 2)
    for j in range(18):
        if body[j,2] > conf_thr:
            cv2.circle(frame, pt(body[j]), 4, _COLORS_BGR[j], -1)

    # Hands
    for hand in (hl, hr):
        for ei, (a, b) in enumerate(_HAND_EDGES):
            if hand[a,2] > conf_thr and hand[b,2] > conf_thr:
                cv2.line(frame, pt(hand[a]), pt(hand[b]), (200,200,0), 1)
        for j in range(len(hand)):
            if hand[j,2] > conf_thr:
                cv2.circle(frame, pt(hand[j]), 3, (0,200,200), -1)
    return frame


def _put_arabic_text(frame, text, pos, font_size=28, color=(255,255,255)):
    """Render Arabic text onto a BGR OpenCV frame using Pillow (bidi-aware)."""
    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)
    pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                                   font_size)
    except Exception:
        font = ImageFont.load_default()
    # Semi-transparent background bar
    bbox = draw.textbbox(pos, bidi_text, font=font)
    pad = 6
    draw.rectangle(
        [bbox[0]-pad, bbox[1]-pad, bbox[2]+pad, bbox[3]+pad],
        fill=(0, 0, 0, 160)
    )
    draw.text(pos, bidi_text, font=font, fill=color)
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


def render_annotated_video(
    video_path,
    pose_npz_path,
    translation,
    out_path,
    template_idx=1,
    fps=None,
):
    """Generate annotated output video.

    Overlays the OpenPose skeleton and the predicted Arabic translation
    on every frame of the original video.

    Parameters
    ----------
    video_path    : path to the original .mp4
    pose_npz_path : path to the .npz produced by mosl.pose.extract_dataset
                    (keys: pose_keypoints_2d, hand_left_keypoints_2d,
                           hand_right_keypoints_2d)
    translation   : predicted Arabic string
    out_path      : output .mp4 path
    template_idx  : which TEMPLATES entry to use for the overlay text
    fps           : output fps (None = same as input)
    """
    video_path = Path(video_path)
    out_path   = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Load pose data
    z = np.load(pose_npz_path)
    body_kpts = z['pose_keypoints_2d']           # (T, 54)
    hl_kpts   = z['hand_left_keypoints_2d']      # (T, 63)
    hr_kpts   = z['hand_right_keypoints_2d']     # (T, 63)
    n_pose    = body_kpts.shape[0]

    cap = cv2.VideoCapture(str(video_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_fps = fps or src_fps

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(out_path), fourcc, out_fps, (W, H))

    overlay_text = apply_template(translation, template_idx)
    text_pos = (10, H - 60)

    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = min(frame_idx, n_pose - 1)

        # Build flat keypoint array: body(54) + left(63) + right(63) = 180 floats
        pose_flat = np.concatenate([
            body_kpts[t],   # (54,) → 18 joints × (x,y,c)
            hl_kpts[t],     # (63,) → 21 joints × (x,y,c)
            hr_kpts[t],     # (63,)
        ]).reshape(-1, 3)   # (60, 3)

        frame = _draw_skeleton_cv2(frame, pose_flat)
        frame = _put_arabic_text(frame, overlay_text, text_pos)
        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f'Annotated video saved -> {out_path}  ({frame_idx} frames)')
    return out_path


def render_batch(results, video_root, npz_root, out_root, template_idx=1):
    """Render annotated videos for a list of inference results.

    Parameters
    ----------
    results     : list of {clip_id, reference, hypothesis} dicts
    video_root  : root directory containing .mp4 files
    npz_root    : root directory containing .npz keypoint files
    out_root    : output directory for annotated videos
    """
    video_root = Path(video_root)
    npz_root   = Path(npz_root)
    out_root   = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    rendered = []
    for r in tqdm(results, desc='Rendering'):
        cid = r['clip_id']
        vids = list(video_root.rglob(f'{cid}.mp4'))
        npzs = list(npz_root.rglob(f'{cid}.npz'))
        if not vids or not npzs:
            print(f'  [SKIP] {cid}: video={bool(vids)} npz={bool(npzs)}')
            continue
        out = out_root / f'{cid}_annotated.mp4'
        render_annotated_video(vids[0], npzs[0], r['hypothesis'], out,
                               template_idx=template_idx)
        rendered.append(str(out))
    print(f'Rendered {len(rendered)}/{len(results)} videos -> {out_root}')
    return rendered


# ── Demo: synthetic frame test ────────────────────────────────────────────
def _demo_render():
    """Smoke-test the rendering pipeline on a synthetic black frame."""
    H, W = 480, 640
    frame = np.zeros((H, W, 3), dtype=np.uint8)
    # Fake body keypoints: place joints in a rough human shape
    fake_body = np.zeros((18, 3), dtype=np.float32)
    positions = [
        (320,80),(320,160),(280,160),(240,240),(220,320),
        (360,160),(400,240),(420,320),(300,280),(290,360),
        (290,440),(340,280),(350,360),(350,440),(310,70),
        (330,70),(300,90),(340,90),
    ]
    for i,(x,y) in enumerate(positions):
        fake_body[i] = [x, y, 1.0]
    frame = _draw_skeleton_cv2(frame, fake_body)
    text  = apply_template('أَنَا', 1)
    frame = _put_arabic_text(frame, text, (10, H-60))
    out = OUTPUTS_DIR / 'demo_annotated_frame.png'
    cv2.imwrite(str(out), frame)
    # Show in notebook
    plt.figure(figsize=(8,5))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Demo: skeleton overlay + Arabic translation text')
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'demo_render.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Demo frame saved ->', out)

_demo_render()

print()
print('Full pipeline summary:')
print('  Video → OpenPose → RGB features → Fusion → LLM → Text + Annotated Video')
print()
print('To render annotated videos after inference:')
print('  render_batch(results,')
print('      video_root = DATA_DIR / "raw" / "vedios-dataset",')
print('      npz_root   = PROCESSED_DIR / "keypoints_2d",')
print('      out_root   = OUTPUTS_DIR / "annotated_videos")')
